# **1. BIBLIOTECAS**

In [ ]:
import time
import json
import logging
import requests
import pandas as pd
from datetime import datetime, timedelta, UTC
from concurrent.futures import ThreadPoolExecutor, as_completed

# **2. DADOS DE ACESSO**

##**2.1. API Key**

In [ ]:
API_KEY = "vU6PB94IWrRqSCNrMWfy36wB4pHUdMO1FUonfZw9"

##**2.2. Shortname**

In [ ]:
SHORTNAMES = {
  "ubu" : "BRAEO6001",
  "germano" : "BRAEO6002"
}

##**2.3. URL's**

In [ ]:
BASE_URL = "https://analystapi.repcenter.skf.com/"

##**2.4. Header**

In [ ]:
HEADERS = {
    "x-api-key": API_KEY,
    "Content-Type": "application/json"
}

##**2.5. Fields**

###**2.5.1. Assets**

In [ ]:
FIELDS_ASSETS = """
assetId
assetName
assetDescription
functionalLocation
parentId
parentName
criticality
assetStatus
assetSegment
conditionIndex
equipmentType
"""

###**2.5.2. Overall alarms**

In [ ]:
FIELDS_ALARMS = """
assetId
assetName
pointId
pointName
unit
alarmMethod
publicOrPrivateAlarm
alertHigh
alertLow
dangerHigh
dangerLow
"""

###**2.5.3. Measurements**

In [ ]:
FIELDS_MEASUREMENTS = """
assetId
assetName
pointId
pointName
channel
unit
pointStatus
collectedDate
overallValue
"""

###**2.5.4. Conditions**

In [ ]:
FIELDS_CONDITIONS = """
assetId
collectDate
conditionDate
conditionId
conditionState
inspectionType
trend
technique
status
diagnostic
observation
author
workOrder {
    id
}
"""

**Campos existentes não utilizados da api**

globalValue

globalValueUnity

workOrder {
   orderNumber
   deadline
   priority
   technique
   scheduledDate
   openingDate
   reWork
   cmmsRegister
   cmms
   services
   situation
   author
}

intervention {
   date
   interventionType
   description
   isDiagnosticCorrect
}

###**2.5.5. Workorders**

In [ ]:
FIELDS_WORKORDERS = """
assetId
id
deadline
priority
technique
scheduledDate
openingDate
reWork
cmmsRegister
cmms
services
situation
intervention {
    date
    interventionType
    description
    isDiagnosticCorrect
}
"""

**Campos existentes não utilizados da api**

orderNumber

author

##**2.6. Funções**

###**2.6.1. Logger (tempo real)**

In [ ]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s"
)

logger = logging.getLogger(__name__)

###**2.6.2. Log estruturado (memória)**

In [ ]:
LOGS = []

def add_log(status, endpoint, origem, asset_id=None, tentativa=None, tempo=None, mensagem=None):
    LOGS.append({
        "timestamp": datetime.now(UTC).isoformat(timespec="seconds"),
        "endpoint": endpoint,
        "status": status,
        "origem": origem,
        "assetId": asset_id,
        "tentativa": tentativa,
        "tempo": tempo,
        "mensagem": mensagem
    })

###**2.6.3. Retry com Log**

In [ ]:
def request_with_retry(url, payload, origem, endpoint, asset_id=None, retries=5, backoff=2):

    for attempt in range(retries):
        start_time = time.time()

        try:
            response = requests.post(
                url,
                headers=HEADERS,
                json=payload,
                timeout=30
            )

            elapsed = round(time.time() - start_time, 2)

            if response.status_code == 200:
                logger.info(f"[{endpoint}] SUCCESS | origem={origem} asset={asset_id} tempo={elapsed}s")
                add_log("SUCCESS", endpoint, origem, asset_id, attempt+1, elapsed)
                return response

            elif response.status_code in [429, 500, 502, 503, 504]:
                wait = backoff ** attempt

                logger.warning(f"[{endpoint}] RETRY {response.status_code} | origem={origem} tentativa={attempt+1}")
                add_log("RETRY", endpoint, origem, asset_id, attempt+1, elapsed, f"HTTP {response.status_code}")

                time.sleep(wait)

            else:
                logger.error(f"[{endpoint}] ERROR | origem={origem} asset={asset_id} erro={response.text}")
                add_log("ERROR", endpoint, origem, asset_id, attempt+1, mensagem=response.text)
                raise Exception(response.text)

        except Exception as e:
            wait = backoff ** attempt

            logger.warning(f"[{endpoint}] RETRY_NETWORK | origem={origem} tentativa={attempt+1}")
            add_log("RETRY_NETWORK", endpoint, origem, asset_id, attempt+1, mensagem=str(e))

            time.sleep(wait)

    logger.critical(f"[{endpoint}] FAIL | origem={origem} asset={asset_id}")
    add_log("FAIL", endpoint, origem, asset_id, mensagem="Falha após retries")

    raise Exception(f"[{origem}] Falhou após {retries} tentativas")

###**2.6.2. Conversão para datetime**

In [ ]:
def convert_dates(df, columns):
    for col in columns:
        if col in df.columns:
            if df[col].dtype == "object":
                sample = df[col].dropna().astype(str).head(5)

                if any("," in x and "GMT" in x for x in sample):
                    df[col] = pd.to_datetime(df[col], utc=True, errors="coerce")
    return df

###**2.6.3. Remoção de timezone**

In [ ]:
def remove_timezone(df, columns):
    for col in columns:
        if col in df.columns:
            df[col] = df[col].dt.tz_localize(None)
    return df

##**2.7. Sheets**

###**2.7.1. Autenticação no Sheets**

In [ ]:
# bibliotecas do Google
import gspread
from google.colab import auth
from google.auth import default
from gspread_dataframe import get_as_dataframe, set_with_dataframe

# autentica no Google
auth.authenticate_user()

# use 'default' para pegar as credenciais no formato esperado pelo gspread
creds, _ = default()
gc = gspread.authorize(creds)

# **3. REQUISIÇÃO: ASSETS**

##**3.1. Busca com paginação**

In [ ]:
def fetch_assets(shortname, origem):

    url = f"{BASE_URL}/{shortname}/assets"
    all_data = []
    cursor = None
    ENDPOINT = "assets"

    while True:

        expression = f"""
        {{
            filter{f"(cursor: {cursor})" if cursor else "(rowsPerPage: 300)"} {{
                {FIELDS_ASSETS}
            }}
        }}
        """

        payload = {"expression": expression}

        response = request_with_retry(url, payload, origem, ENDPOINT)

        result = response.json()

        data = result.get("data", [])
        next_cursor = result.get("nextCursor")

        if not data:
            logger.warning(f"[{ENDPOINT}] EMPTY | origem={origem}")
            add_log("EMPTY", ENDPOINT, origem)

        for item in data:
            item["origem"] = origem

        all_data.extend(data)

        if not next_cursor:
            break

        cursor = next_cursor

    logger.info(f"[{ENDPOINT}] FINAL | origem={origem} total={len(all_data)}")

    return all_data

##**3.2. Execução paralela**

In [ ]:
def fetch_all_assets_parallel():
    results = []

    with ThreadPoolExecutor(max_workers=2) as executor:
        futures = {
            executor.submit(fetch_assets, short, name): name
            for name, short in SHORTNAMES.items()
        }

        for future in as_completed(futures):
            try:
                data = future.result()
                results.extend(data)
            except Exception as e:
                print(f"Erro em {futures[future]}: {e}")

    return results

##**3.3. Main**

In [ ]:
if __name__ == "__main__":
    ativos = fetch_all_assets_parallel()

    df_assets = pd.DataFrame(ativos)

    print("\nResumo:")
    print(df_assets.groupby("origem").size())

    print("\nTotal geral:", len(df_assets))

##**3.4. DataFrame**

In [ ]:
display(df_assets)

In [ ]:
#download excel
df_assets.to_excel("cda_offline_skf_assets.xlsx", index=False)

##**3.5. Carga no Sheets**

In [ ]:
# Nome da planilha
nome_da_planilha = "cda_offline_skf_assets"
nome_da_aba = "Sheet1"

# Abre a planilha
planilha = gc.open(nome_da_planilha)
aba = planilha.worksheet(nome_da_aba)

# Limpa a aba antes de escrever os dados (opcional)
aba.clear()

# Envia o DataFrame para a aba
set_with_dataframe(aba, df_assets)

print("Dados enviados com sucesso para o Google Sheets!")

#**4. REQUISIÇÃO: OVERALL ALARMS**

##**4.1. Busca com paginação**

In [ ]:
def fetch_overall_alarm(shortname, origem):

    url = f"{BASE_URL}/{shortname}/overall-alarm"
    all_data = []
    cursor = None
    ENDPOINT = "overall-alarm"

    while True:
        if cursor:
            expression = f"""
            {{
                filter(cursor: {cursor}) {{
                    {FIELDS_ALARMS}
                }}
            }}
            """
        else:
            expression = f"""
            {{
                filter {{
                    {FIELDS_ALARMS}
                }}
            }}
            """

        payload = {"expression": expression}

        response = request_with_retry(
            url,
            payload,
            origem,
            endpoint=ENDPOINT
        )

        result = response.json()

        data = result.get("data", [])
        next_cursor = result.get("nextCursor")

        if not data:
            logger.warning(f"[{ENDPOINT}] EMPTY | origem={origem}")

            add_log(
                "EMPTY",
                ENDPOINT,
                origem,
                mensagem="Sem dados retornados"
            )

        for item in data:
            item["origem"] = origem

        all_data.extend(data)

        if not next_cursor:
            break

        cursor = next_cursor

    logger.info(f"[{ENDPOINT}] FINAL | origem={origem} total={len(all_data)}")

    add_log(
        "FINAL",
        ENDPOINT,
        origem,
        mensagem=f"Total registros: {len(all_data)}"
    )

    return all_data

##**4.2. Execução paralela**

In [ ]:
def fetch_all_overall_alarm():
    results = []

    with ThreadPoolExecutor(max_workers=2) as executor:
        futures = {
            executor.submit(fetch_overall_alarm, short, name): name
            for name, short in SHORTNAMES.items()
        }

        for future in as_completed(futures):
            origem = futures[future]

            try:
                data = future.result()
                results.extend(data)
            except Exception as e:
                print(f"Erro em {origem}: {e}")

    return results

##**4.3. Main**

In [ ]:
if __name__ == "__main__":
    dados = fetch_all_overall_alarm()

    if not dados:
        print("Nenhum dado retornado da API")
    else:
        df_overall = pd.DataFrame(dados)

        print("\nResumo:")
        print(df_overall.groupby("origem").size())

        print("\nTotal geral:", len(df_overall))

##**4.4. DataFrame**

In [ ]:
display(df_overall)

In [ ]:
#download excel
df_overall.to_excel("cda_offline_skf_alarms.xlsx", index=False)

##**4.5. Carga no Sheets**

In [ ]:
# Nome da planilha
nome_da_planilha = "cda_offline_skf_alarms"
nome_da_aba = "Sheet1"

# Abre a planilha
planilha = gc.open(nome_da_planilha)
aba = planilha.worksheet(nome_da_aba)

# Limpa a aba antes de escrever os dados (opcional)
aba.clear()

# Envia o DataFrame para a aba
set_with_dataframe(aba, df_overall)

print("Dados enviados com sucesso para o Google Sheets!")

#**5. REQUISIÇÃO: LAST MEASUREMENT**

##**5.1. Busca com paginação**

In [ ]:
def fetch_last_measurement_by_asset(shortname, origem, asset_id):

    if pd.isna(asset_id):
        return []

    try:
        asset_id = int(asset_id)
    except:
        return []

    url = f"{BASE_URL}/{shortname}/lastmeasurement"
    ENDPOINT = "lastmeasurement"

    expression = f"""
    {{
        filter(assetId: {asset_id}) {{
            {FIELDS_MEASUREMENTS}
        }}
    }}
    """

    payload = {"expression": expression}

    response = request_with_retry(
        url,
        payload,
        origem,
        endpoint=ENDPOINT,
        asset_id=asset_id
    )

    result = response.json()
    data = result.get("data", [])

    if not data:
        logger.warning(f"[{ENDPOINT}] EMPTY | origem={origem} asset={asset_id}")
        add_log("EMPTY", ENDPOINT, origem, asset_id)
        return []

    for item in data:
        item["origem"] = origem
        item["assetId"] = asset_id

    return data

##**5.2. Execução paralela**

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed

def fetch_all_last_measurements(df_assets):

    df_assets = df_assets[
        df_assets["assetStatus"].notna() &
        (df_assets["assetStatus"] != "None")
    ]

    results = []

    with ThreadPoolExecutor(max_workers=10) as executor:
        futures = [
            executor.submit(
                fetch_last_measurement_by_asset,
                row["shortname"],
                row["origem"],
                row["assetId"]
            )
            for _, row in df_assets.iterrows()
        ]

        for future in as_completed(futures):
            try:
                data = future.result()
                if data:
                    results.extend(data)
            except Exception as e:
                logger.error(f"Erro: {e}")

    return results

##**5.3. Main**

In [ ]:
if __name__ == "__main__":

    df_assets = df_assets.dropna(subset=["assetId"])
    df_assets["shortname"] = df_assets["origem"].map(SHORTNAMES)

    dados = fetch_all_last_measurements(df_assets)

    if not dados:
        print("Nenhum dado retornado da API")
    else:
        df_last_measurement = pd.DataFrame(dados)

        print("\nResumo:")
        print(df_last_measurement.groupby("origem").size())

        print("\nTotal geral:", len(df_last_measurement))

##**5.4. DataFrame**

In [ ]:
# conversão para datetime
df_last_measurement = convert_dates(
    df_last_measurement,
    ["collectedDate"]
)

# remoção de timezone
df_last_measurement = remove_timezone(
    df_last_measurement,
    ["collectedDate"]
)

display(df_last_measurement)

In [ ]:
#download excel
df_last_measurement.to_excel("cda_offline_skf_lastmeasurements.xlsx", index=False)

##**5.5. Carga no Sheets**

In [ ]:
# Nome da planilha
nome_da_planilha = "cda_offline_skf_lastmeasurements"
nome_da_aba = "Sheet1"

# Abre a planilha
planilha = gc.open(nome_da_planilha)
aba = planilha.worksheet(nome_da_aba)

# Limpa a aba antes de escrever os dados (opcional)
aba.clear()

# Envia o DataFrame para a aba
set_with_dataframe(aba, df_last_measurement)

print("Dados enviados com sucesso para o Google Sheets!")

#**6. REQUISIÇÃO: MEASUREMENT BY PERIOD**

##**6.1. Busca com paginação**

In [ ]:
def fetch_measurements_by_asset(shortname, origem, asset_id, start_date, end_date):

    if pd.isna(asset_id):
        return []

    try:
        asset_id = int(asset_id)
    except:
        return []

    url = f"{BASE_URL}/{shortname}/measurements"
    all_data = []
    cursor = None
    ENDPOINT = "measurements"

    while True:

        expression = f"""
        {{
            filter(
                assetId: {asset_id},
                collectedDateStart: "{start_date}",
                collectedDateEnd: "{end_date}"
                {f", cursor: {cursor}" if cursor else ""}
            ) {{
                {FIELDS_MEASUREMENTS}
            }}
        }}
        """

        payload = {"expression": expression}

        response = request_with_retry(
            url,
            payload,
            origem,
            endpoint=ENDPOINT,
            asset_id=asset_id
        )

        result = response.json()

        data = result.get("data", [])
        next_cursor = result.get("nextCursor")

        for item in data:
            item["origem"] = origem
            item["assetId"] = asset_id

        all_data.extend(data)

        if not next_cursor:
            break

        cursor = next_cursor

    return all_data

##**6.2. Execução paralela**

In [ ]:
def fetch_all_measurements(df_assets, start_date, end_date):

    df_assets = df_assets[
        df_assets["assetStatus"].notna() &
        (df_assets["assetStatus"] != "None")
    ]

    results = []

    with ThreadPoolExecutor(max_workers=4) as executor:
        futures = [
            executor.submit(
                fetch_measurements_by_asset,
                row["shortname"],
                row["origem"],
                row["assetId"],
                start_date,
                end_date
            )
            for _, row in df_assets.iterrows()
        ]

        for future in as_completed(futures):
            try:
                data = future.result()
                if data:
                    results.extend(data)
            except Exception as e:
                logger.error(f"Erro: {e}")

    return results

##**6.3. Main**

In [ ]:
if __name__ == "__main__":

    # carga manual
    #start_date = "2026-02-21 00:00:00"
    #end_date   = "2026-02-21 23:59:59"

    # carga incremental (D-1)
    ontem = datetime.now(UTC) - timedelta(days=1)

    start_date = ontem.strftime("%Y-%m-%d 00:00:00")
    end_date   = ontem.strftime("%Y-%m-%d 23:59:59")

    df_assets = df_assets.dropna(subset=["assetId"])
    df_assets["shortname"] = df_assets["origem"].map(SHORTNAMES)

    dados = fetch_all_measurements(df_assets, start_date, end_date)

    if not dados:
        print("Nenhum dado retornado da API")
    else:
        df_measurements = pd.DataFrame(dados)

        print("\nResumo:")
        print(df_measurements.groupby("origem").size())

        print("\nTotal geral:", len(df_measurements))

##**6.4. DataFrame**

In [ ]:
# conversão para datetime
df_measurements = convert_dates(
    df_measurements,
    ["collectedDate"]
)

# remoção de timezone
df_measurements = remove_timezone(
    df_measurements,
    ["collectedDate"]
)

# ordenação crescente
df_measurements = df_measurements.sort_values(by="collectedDate")

display(df_measurements)

In [ ]:
#download excel
df_measurements.to_excel("cda_offline_skf_measurements.xlsx", index=False)

##**6.5. Carga no Sheets**

In [ ]:
# Nome da planilha e aba
nome_da_planilha = "cda_offline_skf_measurements"
nome_da_aba = "Sheet1"

# Abre a planilha e aba
planilha = gc.open(nome_da_planilha)
aba = planilha.worksheet(nome_da_aba)

# Lê os dados atuais da aba (já existentes)
df_existente = get_as_dataframe(aba, evaluate_formulas=True).dropna(how="all")

# Garante que colunas estão no mesmo formato e ordem
colunas_chave = ['assetId', 'assetName', 'pointId', 'pointName', 'channel', 'unit', 'pointStatus', 'overallValue', 'origem']
df_existente = df_existente[colunas_chave].dropna()

# Remove duplicados e encontra apenas as linhas novas
df_novos = df_measurements[~df_measurements.isin(df_existente.to_dict(orient='list')).all(axis=1)]

# Se houver novos registros, adiciona abaixo
if not df_novos.empty:
    # Número de linhas já existentes (para inserir a partir da próxima linha vazia)
    ultima_linha = len(df_existente) + 2  # +1 para header, +1 para próxima
    set_with_dataframe(aba, df_novos, row=ultima_linha, col=1, include_column_header=False)
    print(f"{len(df_novos)} novas condições adicionadas à planilha!")
else:
    print("Nenhuma condição nova para inserir")

#**7. REQUISIÇÃO: LAST CONDITION**

##**7.1. Busca por ativo**

In [ ]:
def fetch_last_condition(shortname, origem, asset_id):

    url = f"{BASE_URL}/v2/{shortname}/lastcondition"
    ENDPOINT = "lastcondition"

    expression = f"""
    {{
        filter(assetId: {asset_id}) {{
            {FIELDS_CONDITIONS}
        }}
    }}
    """

    payload = {"expression": expression}

    response = request_with_retry(
        url,
        payload,
        origem,
        endpoint=ENDPOINT,
        asset_id=asset_id
    )

    data = response.json().get("data", [])

    if not data:
        logger.warning(f"[{ENDPOINT}] EMPTY | origem={origem} asset={asset_id}")
        add_log("EMPTY", ENDPOINT, origem, asset_id)
        return []

    for item in data:
        item["origem"] = origem
        item["assetId"] = asset_id

    return data

##**7.2. Execução paralela**

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed

def fetch_all_last_condition(df_assets):

    df_assets_ativos = df_assets[
        (df_assets["assetStatus"].notna()) &
        (df_assets["assetStatus"] != "None")
    ]

    results = []

    with ThreadPoolExecutor(max_workers=12) as executor:

        futures = [
            executor.submit(
                fetch_last_condition,
                row["shortname"],
                row["origem"],
                row["assetId"]
            )
            for _, row in df_assets_ativos.iterrows()
        ]

        for future in as_completed(futures):
            try:
                data = future.result()
                if data:
                    results.extend(data)
            except Exception as e:
                logger.error(f"Erro: {e}")

    return results

##**7.3. Main**

In [ ]:
if __name__ == "__main__":

    df_assets = df_assets.dropna(subset=["assetId"])
    df_assets["shortname"] = df_assets["origem"].map(SHORTNAMES)

    dados = fetch_all_last_condition(df_assets)

    if not dados:
        print("Nenhum dado retornado da API")
    else:
        df_last_condition = pd.DataFrame(dados)

        if "workOrder" in df_last_condition.columns:
            df_workorder = pd.json_normalize(df_last_condition["workOrder"])

        print("\nResumo:")
        print(df_last_condition.groupby("origem").size())

        print("\nTotal geral:", len(df_last_condition))

##**7.4. DataFrame**

In [ ]:
# separação de dados aninhados
df_last_condition = df_last_condition.join(
    pd.json_normalize(df_last_condition["workOrder"]).add_prefix("workOrder")
    ).drop(columns=["workOrder"]
)

# conversão para datetime
df_last_condition = convert_dates(
    df_last_condition,
    ["collectDate",
    "conditionDate"]
)

# remoção de timezone
df_last_condition = remove_timezone(
    df_last_condition,
    ["collectDate",
    "conditionDate"]
)

display(df_last_condition)

In [ ]:
#download excel
df_last_condition.to_excel("cda_offline_skf_lastcondition.xlsx", index=False)

##**7.5. Carga no Sheets**

In [ ]:
# Nome da planilha
nome_da_planilha = "cda_offline_skf_lastcondition"
nome_da_aba = "Sheet1"

# Abre a planilha
planilha = gc.open(nome_da_planilha)
aba = planilha.worksheet(nome_da_aba)

# Limpa a aba antes de escrever os dados (opcional)
aba.clear()

# Envia o DataFrame para a aba
set_with_dataframe(aba, df_last_condition)

print("Dados enviados com sucesso para o Google Sheets!")

#**8. REQUISIÇÃO: CONDITION BY PERIOD**

##**8.1. Busca com paginação**

In [ ]:
def fetch_conditions_by_asset(shortname, origem, asset_id, start_date, end_date):

    if pd.isna(asset_id):
        return []

    try:
        asset_id = int(asset_id)
    except:
        return []

    url = f"{BASE_URL}/v2/{shortname}/conditions"
    all_data = []
    cursor = None
    ENDPOINT = "conditions"

    while True:
        if cursor:
            expression = f"""
            {{
                filter(
                    assetId: {asset_id},
                    collectedDateStart: "{start_date}",
                    collectedDateEnd: "{end_date}",
                    cursor: {cursor}
                ) {{
                    {FIELDS_CONDITIONS}
                }}
            }}
            """
        else:
            expression = f"""
            {{
                filter(
                    assetId: {asset_id},
                    collectedDateStart: "{start_date}",
                    collectedDateEnd: "{end_date}"
                ) {{
                    {FIELDS_CONDITIONS}
                }}
            }}
            """

        payload = {"expression": expression}

        response = request_with_retry(
            url,
            payload,
            origem,
            endpoint=ENDPOINT
        )

        result = response.json()

        data = result.get("data", [])
        next_cursor = result.get("nextCursor")

        for item in data:
            item["origem"] = origem
            item["assetId"] = asset_id

        all_data.extend(data)

        if not next_cursor:
            break

        cursor = next_cursor

    logger.info(f"[{ENDPOINT}] FINAL | origem={origem} asset={asset_id} total={len(all_data)}")

    add_log(
        "FINAL",
        ENDPOINT,
        origem,
        asset_id,
        mensagem=f"Total registros: {len(all_data)}"
    )

    return all_data

##**8.2. Execução paralela**

In [ ]:
def fetch_all_conditions(df_assets, start_date, end_date):

    df_assets = df_assets[
        df_assets["assetStatus"].notna() &
        (df_assets["assetStatus"] != "None")
    ]

    results = []

    with ThreadPoolExecutor(max_workers=4) as executor:
        futures = []

        for _, row in df_assets.iterrows():
            futures.append(
                executor.submit(
                    fetch_conditions_by_asset,
                    row["shortname"],
                    row["origem"],
                    row["assetId"],
                    start_date,
                    end_date
                )
            )

        for future in as_completed(futures):
            try:
                results.extend(future.result())
            except Exception as e:
                print(f"Erro: {e}")

    return results

##**8.3. Main**

In [ ]:
if __name__ == "__main__":

    # carga manual
    #start_date = "2026-02-21 00:00:00"
    #end_date   = "2026-02-21 23:59:59"

    # carga incremental (D-1)
    ontem = datetime.now(UTC) - timedelta(days=1)

    start_date = ontem.strftime("%Y-%m-%d 00:00:00")
    end_date   = ontem.strftime("%Y-%m-%d 23:59:59")

    df_assets = df_assets.dropna(subset=["assetId"])
    df_assets["shortname"] = df_assets["origem"].map(SHORTNAMES)

    dados = fetch_all_conditions(df_assets, start_date, end_date)

    if not dados:
        print("Nenhum dado retornado da API")
    else:
        df_conditions = pd.DataFrame(dados)

        print("\nResumo:")
        print(df_conditions.groupby("origem").size())

        print("\nTotal geral:", len(df_conditions))

##**8.4. DataFrame**

In [ ]:
# separação de dados aninhados
df_conditions = df_conditions.join(
    pd.json_normalize(df_conditions["workOrder"]).add_prefix("workOrder_")
    ).drop(columns=["workOrder"]
)

# conversão para datetime
df_conditions = convert_dates(
    df_conditions,
    ["collectDate",
    "conditionDate"]
)

# remoção de timezone
df_conditions = remove_timezone(
    df_conditions,
    ["collectDate",
    "conditionDate"]
)

# ordenação crescente
df_conditions = df_conditions.sort_values(by="conditionDate")

display(df_conditions)

In [ ]:
#download excel
df_conditions.to_excel("cda_offline_skf_conditions.xlsx", index=False)

## **8.5. Carga no Sheets**

In [ ]:
# Nome da planilha e aba
nome_da_planilha = "cda_offline_skf_conditions"
nome_da_aba = "Sheet1"

# Abre a planilha e aba
planilha = gc.open(nome_da_planilha)
aba = planilha.worksheet(nome_da_aba)

# Lê os dados atuais da aba (já existentes)
df_existente = get_as_dataframe(aba, evaluate_formulas=True).dropna(how="all")

# Garante que colunas estão no mesmo formato e ordem
colunas_chave = ['assetId', 'collectDate', 'conditionDate', 'conditionId', 'conditionState', 'inspectionType', 'trend', 'technique', 'status', 'diagnostic', 'observation', 'author', 'workOrder', 'origem']
df_existente = df_existente[colunas_chave].dropna()

# Remove duplicados e encontra apenas as linhas novas
df_novos = df_conditions[~df_conditions.isin(df_existente.to_dict(orient='list')).all(axis=1)]

# Se houver novos registros, adiciona abaixo
if not df_novos.empty:
    # Número de linhas já existentes (para inserir a partir da próxima linha vazia)
    ultima_linha = len(df_existente) + 2  # +1 para header, +1 para próxima
    set_with_dataframe(aba, df_novos, row=ultima_linha, col=1, include_column_header=False)
    print(f"{len(df_novos)} novas condições adicionadas à planilha!")
else:
    print("Nenhuma condição nova para inserir")

#**9. REQUISIÇÃO: WORKORDER BY PERIOD**

##**9.1. Busca com paginação**

In [ ]:
def fetch_workorders(shortname, origem, start_date, end_date):

    url = f"{BASE_URL}/{shortname}/workorders"
    all_data = []
    cursor = None
    ENDPOINT = "workorders"

    while True:

        expression = f"""
        {{
            filter(
                openingDateStart: "{start_date}",
                openingDateEnd: "{end_date}"
                {f", cursor: {cursor}" if cursor else ""}
            ) {{
                {FIELDS_WORKORDERS}
            }}
        }}
        """

        payload = {"expression": expression}

        response = request_with_retry(url, payload, origem, ENDPOINT)

        result = response.json()

        data = result.get("data", [])
        next_cursor = result.get("nextCursor")

        if not data:
            logger.warning(f"[{ENDPOINT}] EMPTY | origem={origem}")
            add_log("EMPTY", ENDPOINT, origem)

        for item in data:
            item["origem"] = origem

        all_data.extend(data)

        if not next_cursor:
            break

        cursor = next_cursor

    logger.info(f"[{ENDPOINT}] FINAL | origem={origem} total={len(all_data)}")

    return all_data

##**9.2. Execução paralela**

In [ ]:
def fetch_all_workorders(start_date, end_date):
    results = []

    with ThreadPoolExecutor(max_workers=2) as executor:
        futures = {
            executor.submit(fetch_workorders, short, name, start_date, end_date): name
            for name, short in SHORTNAMES.items()
        }

        for future in as_completed(futures):
            origem = futures[future]

            try:
                results.extend(future.result())
            except Exception as e:
                print(f"Erro em {origem}: {e}")

    return results

##**9.3. Main**

In [ ]:
if __name__ == "__main__":

    # carga completa
    ontem = datetime.now(UTC) - timedelta(days=1)

    start_date = "2020-01-01 00:00:00"
    end_date   = ontem.strftime("%Y-%m-%d 23:59:59")

    dados = fetch_all_workorders(start_date, end_date)

    if not dados:
        print("Nenhum dado retornado da API")
    else:
        df_workorders = pd.DataFrame(dados)

        print("\nResumo:")
        print(df_workorders.groupby("origem").size())

        print("\nTotal geral:", len(df_workorders))

##**9.4. DataFrame**

In [ ]:
# separação de dados aninhados
df_workorders = df_workorders.join(
    pd.json_normalize(df_workorders["intervention"]).add_prefix("intervention_")
    ).drop(columns=["intervention"])

# conversão para datetime
df_workorders = convert_dates(
    df_workorders,
    ["scheduledDate",
    "openingDate",
     "intervention_date"]
)

# remoção de timezone
df_workorders = remove_timezone(
    df_workorders,
    ["scheduledDate",
    "openingDate",
     "intervention_date"]
)

# ordenação crescente
df_workorders = df_workorders.sort_values(by="openingDate")

display(df_workorders)

In [ ]:
#download excel
df_workorders.to_excel("cda_offline_skf_notes.xlsx", index=False)

## **9.5. Carga no Sheets**

In [ ]:
# Nome da planilha
nome_da_planilha = "cda_offline_skf_notes"
nome_da_aba = "Sheet1"

# Abre a planilha
planilha = gc.open(nome_da_planilha)
aba = planilha.worksheet(nome_da_aba)

# Limpa a aba antes de escrever os dados
aba.clear()

# Envia o DataFrame para a aba
set_with_dataframe(aba, df_workorders)

print("Dados enviados com sucesso para o Google Sheets!")

#**10. LOG**

In [ ]:
df_log = pd.DataFrame(LOGS)

output_log = {
    "resumo": df_log["status"].value_counts().to_dict(),
    "erros": df_log[df_log["status"].isin(["ERROR", "FAIL"])].to_dict(orient="records"),
    "total": len(df_log),
    "logs": LOGS
}

with open("log_execucao.json", "w") as f:
    json.dump(output_log, f, indent=4)

print("Log gerado")

# **11. PSEUDOCÓDIGO**

# CDA Offline SKF

---

## 1. CONFIGURAÇÕES INICIAIS

```
DEFINIR API_KEY
DEFINIR SHORTNAMES = { "ubu": "BRAEO6001", "germano": "BRAEO6002" }
DEFINIR BASE_URL = "https://analystapi.repcenter.skf.com/"
DEFINIR HEADERS = { "x-api-key": API_KEY, "Content-Type": "application/json" }

DEFINIR FIELDS_ASSETS        (campos: assetId, assetName, criticality, conditionIndex, ...)
DEFINIR FIELDS_ALARMS        (campos: assetId, pointId, alarmMethod, alertHigh, dangerLow, ...)
DEFINIR FIELDS_MEASUREMENTS  (campos: assetId, pointId, unit, collectedDate, overallValue, ...)
DEFINIR FIELDS_CONDITIONS    (campos: assetId, conditionState, diagnostic, workOrder.id, ...)
DEFINIR FIELDS_WORKORDERS    (campos: assetId, deadline, priority, intervention.date, ...)

DEFINIR LOGS = []  // lista global para registro de eventos
```

---

## 2. FUNÇÕES UTILITÁRIAS

### 2.1. add_log(status, endpoint, origem, ...)
```
FUNÇÃO add_log(status, endpoint, origem, asset_id, tentativa, tempo, mensagem):
    ADICIONAR ao LOGS:
        timestamp = data/hora atual (UTC)
        endpoint, status, origem, assetId, tentativa, tempo, mensagem
```

### 2.2. request_with_retry(url, payload, ...)
```
FUNÇÃO request_with_retry(url, payload, origem, endpoint, asset_id, retries=5, backoff=2):

    PARA tentativa DE 0 ATÉ retries-1:
        registrar tempo de início

        TENTAR:
            resposta = POST(url, headers=HEADERS, json=payload, timeout=30)
            tempo_decorrido = agora - início

            SE resposta.status == 200:
                registrar log de SUCESSO
                RETORNAR resposta

            SENÃO SE status EM [429, 500, 502, 503, 504]:
                registrar log de RETRY
                aguardar backoff^tentativa segundos
                CONTINUAR loop

            SENÃO:
                registrar log de ERRO
                LANÇAR exceção com texto do erro

        EXCETO qualquer erro de rede:
            registrar log de RETRY_NETWORK
            aguardar backoff^tentativa segundos
            CONTINUAR loop

    registrar log FAIL (esgotou tentativas)
    LANÇAR exceção "Falhou após N tentativas"
```

### 2.3. convert_dates(df, colunas)
```
FUNÇÃO convert_dates(df, colunas):
    PARA CADA coluna EM colunas:
        SE coluna existe no df E é texto:
            SE amostra dos valores contém "," e "GMT":
                converter coluna para datetime com timezone UTC
    RETORNAR df
```

### 2.4. remove_timezone(df, colunas)
```
FUNÇÃO remove_timezone(df, colunas):
    PARA CADA coluna EM colunas:
        SE coluna existe no df:
            remover informação de timezone da coluna
    RETORNAR df
```

---

## 3. ASSETS

### Busca paginada por origem
```
FUNÇÃO fetch_assets(shortname, origem):
    url = BASE_URL + shortname + "/assets"
    cursor = NULO
    all_data = []

    LOOP:
        SE cursor existe:
            expressão = filtro com cursor
        SENÃO:
            expressão = filtro com rowsPerPage=300

        resposta = request_with_retry(url, expressão, ...)
        data, next_cursor = extrair de resposta.json()

        SE data vazia: registrar log EMPTY
        adicionar "origem" em cada item
        all_data += data

        SE next_cursor não existe: SAIR loop
        cursor = next_cursor

    registrar log FINAL com total
    RETORNAR all_data
```

### Execução paralela (2 threads)
```
FUNÇÃO fetch_all_assets_parallel():
    results = []
    PARALELAMENTE (2 threads):
        PARA CADA (nome, shortname) EM SHORTNAMES:
            executar fetch_assets(shortname, nome)
    AGREGAR resultados
    RETORNAR results
```

### Main Assets
```
ativos = fetch_all_assets_parallel()
df_assets = DataFrame(ativos)
EXIBIR resumo por origem e total
SALVAR Excel "cda_offline_skf_assets.xlsx"
ENVIAR df_assets para Google Sheets "cda_offline_skf_assets"
```

---

## 4. OVERALL ALARMS

### Busca paginada por origem
```
FUNÇÃO fetch_overall_alarm(shortname, origem):
    url = BASE_URL + shortname + "/overall-alarm"
    cursor = NULO; all_data = []

    LOOP:
        montar expressão (com ou sem cursor)
        resposta = request_with_retry(url, expressão, ...)
        data, next_cursor = extrair de resposta.json()

        SE data vazia: registrar log EMPTY
        adicionar "origem" em cada item
        all_data += data

        SE next_cursor não existe: SAIR loop
        cursor = next_cursor

    registrar log FINAL
    RETORNAR all_data
```

### Execução paralela e Main
```
FUNÇÃO fetch_all_overall_alarm():
    PARALELAMENTE (2 threads): executar fetch_overall_alarm por shortname
    RETORNAR resultados agregados

df_overall = DataFrame(resultados)
SALVAR Excel "cda_offline_skf_alarms.xlsx"
ENVIAR para Google Sheets "cda_offline_skf_alarms"
```

---

## 5. LAST MEASUREMENT (por ativo)

### Busca por ativo
```
FUNÇÃO fetch_last_measurement_by_asset(shortname, origem, asset_id):
    SE asset_id inválido: RETORNAR []

    url = BASE_URL + shortname + "/lastmeasurement"
    expressão = filtro por assetId

    resposta = request_with_retry(url, expressão, ...)
    data = extrair de resposta.json()

    SE data vazia: registrar log EMPTY; RETORNAR []

    adicionar "origem" e "assetId" em cada item
    RETORNAR data
```

### Execução paralela (10 threads)
```
FUNÇÃO fetch_all_last_measurements(df_assets):
    FILTRAR df_assets: manter apenas ativos com assetStatus válido

    PARALELAMENTE (10 threads):
        PARA CADA ativo no df_assets:
            executar fetch_last_measurement_by_asset(shortname, origem, assetId)

    RETORNAR resultados agregados
```

### Main Last Measurement
```
df_assets["shortname"] = mapear origem → shortname
dados = fetch_all_last_measurements(df_assets)
df_last_measurement = DataFrame(dados)
CONVERTER colunas de data (collectedDate)
REMOVER timezone
SALVAR Excel "cda_offline_skf_lastmeasurements.xlsx"
ENVIAR para Google Sheets "cda_offline_skf_lastmeasurements"
```

---

## 6. MEASUREMENTS POR PERÍODO

### Busca paginada por ativo e período
```
FUNÇÃO fetch_measurements_by_asset(shortname, origem, asset_id, start_date, end_date):
    SE asset_id inválido: RETORNAR []

    url = BASE_URL + shortname + "/measurements"
    cursor = NULO; all_data = []

    LOOP:
        expressão = filtro por assetId + datas + cursor (se existir)
        resposta = request_with_retry(url, expressão, ...)
        data, next_cursor = extrair

        adicionar "origem" e "assetId" em cada item
        all_data += data

        SE next_cursor não existe: SAIR loop
        cursor = next_cursor

    RETORNAR all_data
```

### Main Measurements por Período
```
// Carga incremental: período = ontem (D-1)
start_date = ontem 00:00:00
end_date   = ontem 23:59:59

PARALELAMENTE (4 threads): executar fetch_measurements_by_asset por ativo
df_measurements = DataFrame(resultados)
CONVERTER e REMOVER timezone de "collectedDate"
ORDENAR por "collectedDate" crescente
SALVAR Excel "cda_offline_skf_measurements.xlsx"

// Carga incremental no Sheets (sem duplicar):
df_existente = ler dados atuais do Sheets
df_novos = df_measurements - df_existente (apenas registros novos)
SE df_novos não vazio:
    INSERIR df_novos abaixo da última linha existente
SENÃO:
    IMPRIMIR "Nenhum dado novo"
```

---

## 7. LAST CONDITION (por ativo)

### Busca por ativo
```
FUNÇÃO fetch_last_condition(shortname, origem, asset_id):
    url = BASE_URL + "v2/" + shortname + "/lastcondition"
    expressão = filtro por assetId

    resposta = request_with_retry(url, expressão, ...)
    data = extrair de resposta.json()

    SE data vazia: registrar log EMPTY; RETORNAR []
    adicionar "origem" e "assetId" em cada item
    RETORNAR data
```

### Main Last Condition
```
PARALELAMENTE (12 threads): executar fetch_last_condition por ativo ativo
df_last_condition = DataFrame(resultados)
EXPANDIR coluna "workOrder" (JSON aninhado) → prefixo "workOrder"
CONVERTER e REMOVER timezone de "collectDate", "conditionDate"
SALVAR Excel "cda_offline_skf_lastcondition.xlsx"
ENVIAR para Google Sheets "cda_offline_skf_lastcondition"
```

---

## 8. CONDITIONS POR PERÍODO

### Busca paginada por ativo e período
```
FUNÇÃO fetch_conditions_by_asset(shortname, origem, asset_id, start_date, end_date):
    SE asset_id inválido: RETORNAR []

    url = BASE_URL + "v2/" + shortname + "/conditions"
    cursor = NULO; all_data = []

    LOOP:
        expressão = filtro por assetId + datas + cursor (se existir)
        resposta = request_with_retry(url, expressão, ...)
        data, next_cursor = extrair

        adicionar "origem" e "assetId" em cada item
        all_data += data

        SE next_cursor não existe: SAIR loop
        cursor = next_cursor

    registrar log FINAL
    RETORNAR all_data
```

### Main Conditions por Período
```
// Carga incremental (D-1)
start_date = ontem 00:00:00
end_date   = ontem 23:59:59

PARALELAMENTE (4 threads): executar fetch_conditions_by_asset por ativo
df_conditions = DataFrame(resultados)
EXPANDIR coluna "workOrder" → prefixo "workOrder_"
CONVERTER e REMOVER timezone de "collectDate", "conditionDate"
ORDENAR por "conditionDate" crescente
SALVAR Excel "cda_offline_skf_conditions.xlsx"

// Carga incremental no Sheets (sem duplicar):
df_existente = ler dados atuais do Sheets
df_novos = df_conditions - df_existente
SE df_novos não vazio: INSERIR abaixo da última linha
SENÃO: IMPRIMIR "Nenhum dado novo"
```

---

## 9. WORKORDERS POR PERÍODO

### Busca paginada por origem
```
FUNÇÃO fetch_workorders(shortname, origem, start_date, end_date):
    url = BASE_URL + shortname + "/workorders"
    cursor = NULO; all_data = []

    LOOP:
        expressão = filtro por datas de abertura + cursor (se existir)
        resposta = request_with_retry(url, expressão, ...)
        data, next_cursor = extrair

        SE data vazia: registrar log EMPTY
        adicionar "origem" em cada item
        all_data += data

        SE next_cursor não existe: SAIR loop
        cursor = next_cursor

    registrar log FINAL
    RETORNAR all_data
```

### Main Workorders
```
// Carga completa: de 2020-01-01 até ontem
start_date = "2020-01-01 00:00:00"
end_date   = ontem 23:59:59

PARALELAMENTE (2 threads): executar fetch_workorders por shortname
df_workorders = DataFrame(resultados)
EXPANDIR coluna "intervention" → prefixo "intervention_"
CONVERTER e REMOVER timezone de "scheduledDate", "openingDate", "intervention_date"
ORDENAR por "openingDate" crescente
SALVAR Excel "cda_offline_skf_notes.xlsx"
ENVIAR para Google Sheets "cda_offline_skf_notes" (substitui tudo)
```

---

## 10. LOG FINAL

```
df_log = DataFrame(LOGS)

output_log = {
    "resumo"  : contagem de status (SUCCESS, RETRY, ERROR, FAIL, ...)
    "erros"   : apenas registros com status ERROR ou FAIL
    "total"   : total de entradas no log
    "logs"    : lista completa de LOGS
}

SALVAR output_log em "log_execucao.json"
IMPRIMIR "Log gerado"
```

---

## FLUXO GERAL (visão macro)

```
INÍCIO
│
├─ Configurar credenciais, URLs e campos da API
├─ Autenticar no Google Sheets
│
├─ [3] Buscar ASSETS         → df_assets         → Excel + Sheets
├─ [4] Buscar ALARMS         → df_overall        → Excel + Sheets
├─ [5] Buscar LAST MEASURE   → df_last_measurement → Excel + Sheets
├─ [6] Buscar MEASURES D-1   → df_measurements   → Excel + Sheets (incremental)
├─ [7] Buscar LAST CONDITION → df_last_condition → Excel + Sheets
├─ [8] Buscar CONDITIONS D-1 → df_conditions     → Excel + Sheets (incremental)
├─ [9] Buscar WORKORDERS     → df_workorders     → Excel + Sheets
│
└─ [10] Gerar log_execucao.json
FIM
```